# EDA/SDA 2 (Lean): Which Listings We Keep, Tag or Hide

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

`EDA_SDA_1_Lean` cleaned the full file, compared every field and cut the scope: **the roles TripleTen trains for, plus adjacent tech roles** (about 0.9M listings). This notebook decides, listing by listing, what the app does with each one: **keep, tag or hide.** A fourth treatment, **drop**, happens before it, in `EDA_SDA_1_Lean`'s scope cut.

- **Keep:** the listing is in the table and shown as it is.
- **Tag:** it stays visible with a tag (for example "posted by a recruiter" or "senior"), and the app can rank it lower or offer an opt-in filter. We use a tag when the rule is a keyword rule nobody has hand-checked, or when the person should decide (like a job level).
- **Hide:** the app doesn't show it by default. Only a rule we trust hides.
- **Drop:** the listing is out of scope (not a tech path a career changer can take) and is removed from the table, so it can't confound anything. Only whole occupations, identified by their O*NET code, are dropped; a keyword rule never drops.

Everything else stays in the data. The notebook also shows which role buckets are big enough for their own analysis, and the **2026 market by role bucket and TripleTen program** (the retraining compass's 2026 side). It saves its own final table for `EDA_SDA_3.ipynb` (text), then `EDA_SDA_4.ipynb` (statistics), `TimeSeries`, `ML_Fit` and `ML_Rank`.

**Input from EDA_SDA_1_Lean:** the in-scope table (`analysis_table.parquet`) and company activity in any field (`company_activity.parquet`).

**Conventions:**
- Every rule is written in plain pandas and recorded with its reasoning and the evidence for it.
- Aggregates only (shares, medians, counts); never rows.
- Each table or chart group ends with **What the data shows**: up to three bullets.
- How listings behave once they are kept (company volume, source, freshness, pay disclosure, reposts, missing values) is answered in `EDA_SDA_1_Lean` (sections 5 and 6) and tested with the confounders held fixed in `EDA_SDA_4`. It is not repeated here.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, how does tech compare to other fields, and what's in scope? | Scope table, descriptions, occupation summary, company activity |
| **EDA_SDA_2** | **Which scope listings do we keep, and how do they behave?** | **Scope table, non-text features** |
| EDA_SDA_3 | What do the descriptions say? Skills mapped to O*NET | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

### Clarifying questions
- Which long-open listings are ghost listings, and which are employers still trying to hire?
- Which listings do we tag as posted by a staffing agency or recruiter, and are fast-closing recruiters just staffing agencies?
- How much of the scope is left once the hiding rules are applied, and how much more would an opt-in text tag hide?
- Which role buckets have enough listings and enough career changers for their own analysis?
- What does the 2026 market look like by role bucket and TripleTen program?

### Roadmap
1. **Setup:** load the in-scope table and company activity
2. **The rules at a glance:** one table of every drop, hide, tag and keep rule
3. **Hide:** likely ghosts
4. **Tag:** manager level and above, open 90+ days, staffing agencies, senior titles, the text rules (years asked, degree)
5. **Keep:** the role buckets, and how much of the scope the rules leave
6. **The 2026 market** by role bucket and program (the compass's 2026 side)
7. **The final table**, saved for EDA_SDA_3
8. **Conclusion and next steps**

### About this Lean draft

Written from the question above, in plain pandas (`groupby`, `value_counts`, `merge`, `str.contains`, simple masks), the way `EDA_SDA_1_Lean` was. It is **not** a port of `EDA_SDA_2`: the original also studied how listings behave, built the app's filter fields and modeling features, and screened 68 candidate features. Those are left to the notebooks that need them (`docs/lean_conventions.md`, "The goal"). The faithful port that reproduced the original is commit `df93cc7`.

It reads `EDA_SDA_1_Lean`'s tables from `data/clean/lean/` and writes its own there, not beside `EDA_SDA_2`'s, until it replaces the original.

## 1. Setup

In [ ]:
# 1a Libraries and the project's own code
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Add analysis/ to Python's search path, so "from src..." imports work from inside notebooks/
PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "pl_data.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))
from src import kaplan_meier as km  # the survival method (docs/kaplan_meier.md)
from src.career_switch import MARKETING_BUCKET, marketing_specialist  # the role-bucket rule for O*NET 13-1161
from src.pl_data import DATA_DIR, load_flat  # the project's data loader

pd.set_option("display.max_rows", 200)   # long summary tables show in full
pd.set_option("display.max_colwidth", 140)

CLEAN_DIR = DATA_DIR / "clean"
LEAN_DIR = CLEAN_DIR / "lean"  # EDA_SDA_1_Lean's tables are read from here, and this notebook writes its own here too

# The chart style every chart below uses (the same as EDA_SDA_1_Lean's). ORANGE is reserved for the one group being highlighted.
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#c9c8c2"
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "font.size": 10.5, "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

The in-scope table has about 70 columns; the rules below use these. Loading only them keeps the table small enough for an 8 GB laptop.

| Used for | Columns |
|:--|:--|
| Which listing it is | `id`, `company_id`, `title` |
| When | `first_seen`, `last_seen`, `posted`, `month`, `full_week` (first seen in a full week, not the partial first or last week), `partial_month` |
| Whether it is still open | `live` (last seen within 2 days of September 30), `days_open` |
| What it is | `onet_code`, `role_bucket`, `tripleten_program`, `seniority`, `source_type` |
| What it says | `years_required`, `degree_required`, `equivalent_experience`, `entry_wording`, `changer_wording`, `career_open`, `mentions_remote` |
| Pay | `salary_hourly_usd` |
| Whether it is a repost, and who posted it | `is_repost`, `recruiter_key` (an anonymous key per recruiter, never a name) |

In [ ]:
# 1b Load the in-scope table (EDA_SDA_1_Lean's section 7) and company activity (its section 3)
COLUMNS = ["id", "company_id", "title", "first_seen", "last_seen", "posted", "month", "full_week", "partial_month", "live", "days_open",
           "onet_code", "role_bucket", "tripleten_program", "seniority", "source_type", "years_required", "degree_required",
           "equivalent_experience", "entry_wording", "changer_wording", "career_open", "mentions_remote", "salary_hourly_usd", "is_repost", "recruiter_key"]
jobs = pd.read_parquet(LEAN_DIR / "analysis_table.parquet", columns=COLUMNS)

# True / false / unknown flags come back from Parquet as plain objects; make them pandas' nullable booleans (blank stays blank)
for flag in ["degree_required", "equivalent_experience", "entry_wording", "changer_wording", "career_open", "mentions_remote"]:
    jobs[flag] = jobs[flag].astype("boolean")

# For each company, its newest listing in ANY field (not only the scope): used to tell if a company is still posting
company_activity = pd.read_parquet(LEAN_DIR / "company_activity.parquet")
company_newest = company_activity.newest_listing_any_field
company_newest.index = company_newest.index.astype(str)

DATA_END = pd.Timestamp("2026-09-30", tz="UTC")  # the data's last day
print(f"{len(jobs):,} listings x {jobs.shape[1]} columns, {jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

In [ ]:
# 1c The scope we start from
print("1c The scope from EDA_SDA_1_Lean (nothing is cut here)")
display(pd.Series({
    "listings": f"{len(jobs):,}",
    "original listings (not reposts)": f"{(~jobs.is_repost).sum():,}",
    "companies": f"{jobs.company_id.nunique():,}",
    "still open on September 30": f"{jobs.live.mean():.1%}",
    "under a TripleTen program": f"{jobs.tripleten_program.ne('adjacent tech role').mean():.1%}",
}).to_frame("value"))

## 2. The rules at a glance

Each rule below answers one question about a listing. Sections 3 to 5 define each rule in code, show what it catches, and give the evidence for the treatment. The two **drop** rules are applied upstream, in `EDA_SDA_1_Lean`'s scope cut next to credentialed math (actuaries and mathematicians), so they are listed here and not built here.

In [ ]:
# 2 The rules, in one place. Sections 3 to 7 build each rule's column from this list, and 7.1 and 7.2 read it back.
RULES = [
    {"rule": "Document management specialist", "treatment": "Drop", "column": None, "section": "EDA_SDA_1_Lean 4c",
     "definition": "Its O*NET code is 15-1299.03: records work, not a tech path"},
    {"rule": "GIS technologist", "treatment": "Drop", "column": None, "section": "EDA_SDA_1_Lean 4c",
     "definition": "Its O*NET code is 15-1299.02: mapping work; no TripleTen program trains for it and no career changer in the résumé data moved into it"},
    {"rule": "Likely ghost", "treatment": "Hide", "column": "likely_ghost", "section": "3",
     "definition": "It is still open on September 30, was first seen 90+ days earlier, and its company has posted nothing new in 60 days (or its posting date is over a year before it was first seen)"},
    {"rule": "Manager level or above", "treatment": "Tag (the app's job-level filter; today the app hides it by default for applicants without management experience)",
     "column": "manager_or_above", "section": "4.1",
     "definition": "Its seniority is manager, director, head, vice president, president, c-level, partner or founder"},
    {"rule": "Open 90+ days", "treatment": "Tag, and rank lower", "column": "open_90_days", "section": "4.2",
     "definition": "Still open, first seen 90+ days earlier, and not a likely ghost"},
    {"rule": "Staffing agency or recruiter", "treatment": "Tag", "column": "is_staffing", "section": "4.3",
     "definition": "Its company's name, website or about text names staffing or recruiting"},
    {"rule": "Senior title", "treatment": "Tag, and rank lower", "column": "senior_title", "section": "4.4",
     "definition": "The title says senior, lead, principal, staff or architect"},
    {"rule": "Asks 5+ years", "treatment": "Tag, opt-in filter", "column": "asks_5_plus_years", "section": "4.5",
     "definition": "It states 5 or more years of experience"},
    {"rule": "Degree required, no equivalent", "treatment": "Tag, opt-in filter", "column": "degree_no_equivalent", "section": "4.5",
     "definition": "It requires a degree and doesn't accept equivalent experience"},
    {"rule": "Marketing specialist", "treatment": "Tag (a role bucket), opt-in filter", "column": "marketing_specialist", "section": "5.1",
     "definition": "O*NET 13-1161 with a title that names no analysis or research. Out of scope in principle (it is not what the Data Analytics program trains for), but the split is a title keyword rule, so it tags until the rule is hand-checked"},
]
print("2 The rules at a glance (everything else is kept)")
display(pd.DataFrame(RULES).rename(columns={"rule": "Rule", "treatment": "Treatment", "definition": "A listing matches when", "section": "Section"})
        [["Rule", "Treatment", "A listing matches when", "Section"]].style.hide(axis="index"))

**Why four treatments.** *Drop* is for whole occupations that aren't a tech path, found by an exact O*NET code, so the rule can't misfire. *Hide* is for the one rule we trust but can't prove from a code alone (a ghost listing can't be observed; the 90-day and 60-day cutoffs are choices). *Tag* is for the keyword rules nobody hand-checked, and for anything the person should decide (their own level). A wrong tag costs a label; a wrong drop or hide costs a listing a career changer might have wanted.

## 3. Hide: likely ghosts

A long-open listing is one of two things. It may be a **ghost listing**: forgotten, or no longer watched by the employer. Or it may belong to an employer that is **still trying to hire**. Age alone can't tell them apart, so we look at what else we know about the listing.

**The rule** (decided 2026-10-10). A listing is a **likely ghost** when it is still open on September 30, was first seen more than 90 days earlier, **and** either its company has posted nothing new in 60 days or its own posting date is more than a year before it was first seen. The app hides likely ghosts and says how many it hid and why. A listing that is open 90+ days but doesn't meet the second part isn't hidden; it is tagged "open 90+ days" (section 4).

**The cutoffs are choices, not measurements.** Nobody can observe a ghost, so 90 days and 60 days can't be proven right. 3a shows how listings actually age, so you can see why 90 days is a sensible line and that the curve has no natural break to point at instead.

In [ ]:
# 3a How long do listings stay open, and does closing slow down past 90 days? (Kaplan-Meier, original listings)
# Kaplan-Meier gives the share of listings still open after t days, counting a listing that is still open on September 30
# as unfinished rather than as closed (docs/kaplan_meier.md). Only listings first seen in April-May can be followed for
# 120+ days in this window, so the curve uses them.
originals = jobs[~jobs.is_repost]
early = originals[originals.first_seen < pd.Timestamp("2026-06-01", tz="UTC")]
curve = km.km_curve(early.days_open, ~early.live)

closing_by_age = {}
for t in [0, 30, 60, 90, 120]:
    open_at_start = km.survival_at(curve, t)
    open_30_days_later = km.survival_at(curve, t + 30)
    closing_by_age[f"day {t} to {t + 30}"] = {"still open at the start": open_at_start,
                                              "of those, share that close in the next 30 days": 1 - open_30_days_later / open_at_start}
closing_by_age = pd.DataFrame(closing_by_age).T
print(f"3a Closing by listing age ({len(early):,} listings first seen in April-May)")
display(closing_by_age.style.format("{:.1%}"))

# The same thing as a chart: the share of still-open listings that close in each 30-day stretch of their life
fig, ax = plt.subplots(figsize=(7, 3.6))
share_closing = 100 * closing_by_age["of those, share that close in the next 30 days"]
ax.bar(closing_by_age.index, share_closing, color=[BLUE if t < 90 else ORANGE for t in [0, 30, 60, 90, 120]], width=0.6)
ax.bar_label(ax.containers[0], fmt="%.0f%%", color=INK_2, fontsize=9)
ax.set(title="Share of open listings that close in the next 30 days", xlabel="listing age (days)", ylabel="%")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
# 3b What else do we know about a long-open listing? Three facts, each known for every listing:
#   stale          still open on September 30 and first seen more than 90 days earlier
#   company_quiet  the company's newest listing (in ANY field, from EDA_SDA_1_Lean's company activity) is over 60 days old
#   posted_old     the employer's own posting date is more than a year before the PL crawler first saw the listing
STALE_DAYS, QUIET_DAYS = 90, 60
jobs["stale"] = jobs.live & ((DATA_END - jobs.first_seen).dt.days > STALE_DAYS)
days_since_company_posted = (DATA_END - jobs.company_id.map(company_newest)).dt.days
jobs["company_quiet"] = ~(days_since_company_posted <= QUIET_DAYS)  # a company with no activity record counts as quiet
jobs["posted_old"] = jobs.posted < jobs.first_seen - pd.Timedelta(days=365)

# Among listings still open on September 30: do the older ones look more like forgotten listings than the younger ones?
originals = jobs[~jobs.is_repost]
open_now = originals[originals.live]
old = open_now[open_now.stale]
young = open_now[~open_now.stale]
signals = pd.DataFrame({
    f"open more than {STALE_DAYS} days ({len(old):,})": [old.company_quiet.mean(), old.posted_old.mean()],
    f"open {STALE_DAYS} days or less ({len(young):,})": [young.company_quiet.mean(), young.posted_old.mean()],
}, index=[f"company quiet for {QUIET_DAYS}+ days", "posting date over a year old"])
print("3b Signals among listings still open on September 30 (original listings)")
display(signals.style.format("{:.1%}"))

In [ ]:
# 3c The rule: a likely ghost is stale AND (its company is quiet OR its posting date is over a year old)
jobs["likely_ghost"] = jobs.stale & (jobs.company_quiet | jobs.posted_old)

originals = jobs[~jobs.is_repost]
print(f"original listings still open on September 30: {originals.live.sum():,} | open more than {STALE_DAYS} days: {originals.stale.sum():,} "
      f"| likely ghosts: {originals.likely_ghost.sum():,} ({originals.likely_ghost.sum() / originals.live.sum():.1%} of open listings, "
      f"{originals.likely_ghost.mean():.1%} of all original listings)")

by_bucket = originals.groupby("role_bucket").agg(open_now=("live", "sum"), open_more_than_90_days=("stale", "sum"),
                                                 likely_ghosts=("likely_ghost", "sum"))
by_bucket["likely ghosts, share of open"] = by_bucket.likely_ghosts / by_bucket.open_now
print("\n3c Open listings, open 90+ days, and likely ghosts, by role bucket (original listings)")
display(by_bucket.sort_values("open_now", ascending=False).style.format(
    {"open_now": "{:,.0f}", "open_more_than_90_days": "{:,.0f}", "likely_ghosts": "{:,.0f}", "likely ghosts, share of open": "{:.1%}"}))

**What the data shows**

- **Closing slows with age but never stops.** Of listings first seen in April-May, 53% close in their first 30 days. After day 60, about a third of the listings still open close in each following 30 days (35%, 29% and 30%). There is no break in the curve to point at, so 90 days is a choice.
- **90 days is a sensible line.** Only 17% of listings are still open at day 90, so a listing open past it has outlived 83% of its peers. 19% of listings open on September 30 are older than 90 days (39,045 of 209,565).
- **Old listings are far more often from quiet companies:** 18.3% of the listings open more than 90 days come from a company with nothing new in 60 days, against 2.4% of the younger open ones. That is the signal that separates "forgotten" from "still hiring".
- **7,229 likely ghosts:** 3.4% of the listings open on September 30, 0.9% of all original listings, and 2.4% to 4.1% of the open listings in every role bucket. The app hides them and says how many.

## 4. Tag

A tag is a column the app can show on a listing, rank by or turn into an opt-in filter. None of these listings is hidden. Each tag below is defined in a few lines and comes with the evidence for it.

| Tag | Section |
|:--|:--|
| Manager level or above | 4.1 |
| Open 90+ days | 4.2 |
| Staffing agency or recruiter | 4.3 |
| Senior title | 4.4 |
| Asks 5+ years, degree required with no equivalent | 4.5 |

### 4.1 Manager level or above

PredictLeads gives every listing a `seniority` label. `level` groups the labels into four levels, and a listing is **manager level or above** when its level is manager or leadership. This is a tag and not a hide because the person decides: a career changer with management experience wants these listings, and the app's job-level filter already lets them choose. Today the app hides the tag by default for an applicant who says they have no management experience.

In [ ]:
# 4.1 Level and the manager-level tag
LEVELS = {"junior": "entry", "mid_senior": "mid_senior", "manager": "manager",
          "director": "leadership", "head": "leadership", "vice_president": "leadership", "president": "leadership",
          "c_level": "leadership", "founder": "leadership", "partner": "leadership"}
jobs["level"] = jobs.seniority.astype(object).map(LEVELS).fillna("not stated")   # a seniority label not in LEVELS (or missing) is "not stated"
jobs["manager_or_above"] = jobs.level.isin(["manager", "leadership"])

# What do the levels ask of a newcomer? Share of listings, and the years of experience asked where a listing states them
originals = jobs[~jobs.is_repost]
by_level = originals.groupby("level").agg(listings=("id", "size"), median_years_asked=("years_required", "median"),
                                          asks_2_years_or_fewer=("years_required", lambda y: (y.dropna() <= 2).mean()))
by_level["share of listings"] = by_level.listings / by_level.listings.sum()
LEVEL_ORDER = [level for level in ["entry", "mid_senior", "manager", "leadership", "not stated"] if level in by_level.index]  # every listing has a label, so "not stated" may not occur
by_level = by_level.loc[LEVEL_ORDER, ["listings", "share of listings", "median_years_asked", "asks_2_years_or_fewer"]]
print("4.1 Level of each listing (original listings)")
display(by_level.style.format({"listings": "{:,.0f}", "share of listings": "{:.1%}", "median_years_asked": "{:.1f}", "asks_2_years_or_fewer": "{:.1%}"}))
open_now = originals[originals.live]
print(f"manager level or above: {originals.manager_or_above.mean():.1%} of original listings, {open_now.manager_or_above.mean():.1%} of the ones still open")

### 4.2 Open 90+ days

A listing that is still open, was first seen more than 90 days ago, and isn't a likely ghost. It stays visible. The tag lets the app rank it below newer listings, because 3a shows that old listings keep closing but at a slower rate.

In [ ]:
# 4.2 Open 90+ days: stale, but not a likely ghost
jobs["open_90_days"] = jobs.stale & ~jobs.likely_ghost
originals = jobs[~jobs.is_repost]
open_now = originals[originals.live]
print(f"4.2 Among the {len(open_now):,} original listings still open on September 30: {open_now.open_90_days.sum():,} are tagged "
      f"open 90+ days ({open_now.open_90_days.mean():.1%}), and {open_now.likely_ghost.sum():,} are likely ghosts ({open_now.likely_ghost.mean():.1%}; hidden)")

### 4.3 Staffing agency or recruiter

A listing posted by a staffing agency or recruiting firm, found from its company: the company's name or website, or its about text, names staffing or recruiting. It's a first-pass keyword rule that nobody has hand-checked (decided 2026-10-13), so it is a tag, and the app can offer it as an opt-in filter. The original also tagged companies from agency-style recruiter titles (33 companies, 0.1% of listings); that rule is left out.

In [ ]:
# 4.3 Staffing agencies: tag listings whose company looks like a staffing or recruiting firm
# The companies table (every company) has each company's name, website and short "about" texts
companies = load_flat("companies", ["id", "attributes.company_name", "attributes.domain", "attributes.description_short", "attributes.meta_title"])
companies = companies.drop_duplicates("id").set_index("id")
companies = companies[companies.index.isin(set(jobs.company_id.dropna()))]   # only the companies in the scope

# Words in a company's name or website that point to a staffing firm (matched at the start of a word), and a few that must
# be a whole word on their own ("talent" and "temp" would otherwise match Tempus, Talentum and so on)
STAFFING_WORDS = ["staffing", "recruit", "personnel", "placement", "headhunt", "manpower", "search group", "search partners",
                  "employment agency", "employment services", "employment solutions", "robert half", "adecco", "randstad",
                  "insight global", "teksystems", "aerotek", "kforce", "apex systems", "cybercoders", "jobot", "allegis",
                  "express employment", "spherion", "akkodis", "beacon hill", "motion recruitment", "kelly services"]
STAFFING_WHOLE_WORDS = ["talent", "temp", "temps", "hays"]
# Phrases in a company's about text that say it is a staffing or recruiting firm
STAFFING_ABOUT = ["staffing agency", "staffing firm", "staffing company", "staffing services", "staffing solutions",
                  "recruiting agency", "recruiting firm", "recruiting company", "recruitment agency", "recruitment firm",
                  "recruitment company", "executive search firm", "placement agency", "placement firm", "temporary staffing",
                  "we place candidates", "we place talent"]

name_text = (companies["attributes.company_name"].fillna("") + " " + companies["attributes.domain"].fillna("")).str.lower()
about_text = (companies["attributes.description_short"].fillna("") + " " + companies["attributes.meta_title"].fillna("")).str.lower()
by_name = (name_text.str.contains(r"\b(?:" + "|".join(STAFFING_WORDS) + ")", regex=True)
           | name_text.str.contains(r"\b(?:" + "|".join(STAFFING_WHOLE_WORDS) + r")\b", regex=True))
by_about = about_text.str.contains("|".join(STAFFING_ABOUT), regex=True)
staffing_companies = companies.index[by_name | by_about]
jobs["is_staffing"] = jobs.company_id.isin(staffing_companies)

print("4.3 Staffing agencies and recruiters")
display(pd.Series({
    "companies in the scope": f"{len(companies):,}",
    "tagged by name or website": f"{by_name.sum():,}",
    "tagged by about text only": f"{(by_about & ~by_name).sum():,}",
    "listings tagged": f"{jobs.is_staffing.sum():,} ({jobs.is_staffing.mean():.1%})",
}).to_frame("value"))

In [ ]:
# 4.3 Evidence: do staffing listings close faster, and are fast-closing recruiters just staffing agencies?
# Median days open (Kaplan-Meier) for the listings of staffing companies vs. employers, and by whether a recruiter is named
jobs["recruiter_group"] = np.select([jobs.recruiter_key.isna(), jobs.recruiter_key.map(jobs.recruiter_key.value_counts()) >= 5],
                                    ["no recruiter named", "recruiter with 5+ listings"], "recruiter with 1-4 listings")


def median_days_open(group):
    """Median days open for a group of listings, counting listings still open on September 30 as unfinished."""
    return km.median_survival(group.days_open, ~group.live)


staffing_window = pd.DataFrame({
    "listings": jobs.groupby("is_staffing").size(),
    "median days open": jobs.groupby("is_staffing")[["days_open", "live"]].apply(median_days_open),
}).rename(index={True: "staffing agency or recruiter", False: "employer"})
print("4.3 Median days open (Kaplan-Meier): staffing agencies vs. employers")
display(staffing_window.style.format({"listings": "{:,.0f}", "median days open": "{:.1f}"}))

recruiter_window = pd.DataFrame({
    "listings": jobs.groupby(["recruiter_group", "is_staffing"]).size(),
    "median days open": jobs.groupby(["recruiter_group", "is_staffing"])[["days_open", "live"]].apply(median_days_open),
}).rename(index={True: "staffing agency", False: "employer"}, level="is_staffing")
print("\n4.3 Median days open (Kaplan-Meier) by recruiter and staffing agency")
display(recruiter_window.style.format({"listings": "{:,.0f}", "median days open": "{:.1f}"}))

### 4.4 Senior title

The title says senior, lead, principal, staff or architect. PredictLeads' `mid_senior` label covers every experienced individual contributor, so it can't separate a senior engineer from a mid-level one; the title can. It's a keyword rule, so it is a tag: the app shows these listings ranked lower with a "senior" tag and doesn't hide them (decided 2026-10-11), because some career changers have senior experience.

In [ ]:
# 4.4 Senior title, and the evidence that senior-titled listings ask more of a newcomer
jobs["senior_title"] = jobs.title.str.lower().str.contains(r"\b(?:senior|sr\.?|lead|principal|staff|architect)\b", regex=True)

# Within PredictLeads' mid_senior label: do senior-titled listings ask more than the others?
mid_senior = jobs[(jobs.level == "mid_senior") & ~jobs.is_repost]
asks = mid_senior.groupby("senior_title").agg(listings=("id", "size"), median_years_asked=("years_required", "median"),
                                              asks_5_plus_years=("years_required", lambda y: (y.dropna() >= 5).mean()),
                                              open_to_career_changers=("career_open", "mean"),
                                              median_hourly_pay=("salary_hourly_usd", "median"))
asks.index = asks.index.map({True: "senior title", False: "other mid_senior"})
print(f"4.4 Senior title: {jobs.senior_title.mean():.1%} of listings. Within mid_senior (original listings):")
display(asks.style.format({"listings": "{:,.0f}", "median_years_asked": "{:.1f}", "asks_5_plus_years": "{:.1%}",
                           "open_to_career_changers": "{:.1%}", "median_hourly_pay": "${:,.2f}"}))

### 4.5 Years asked and degree required

Two text rules from the description: **asks 5+ years** (the most years of experience the description states is 5 or more) and **degree required, no equivalent** (it says a bachelor's, master's or associate degree is required and doesn't accept equivalent experience). They are keyword rules, so they are tags the person can turn into filters ("I have no degree"), and they hide nothing. A listing with no description, or that states no years, isn't tagged: unknown is counted as not tagged, and the table shows how many are unknown.

In [ ]:
# 4.5 The two text tags (unknown is counted as not tagged)
jobs["asks_5_plus_years"] = jobs.years_required >= 5                  # a listing that states no years is False
jobs["degree_no_equivalent"] = (jobs.degree_required & ~jobs.equivalent_experience).fillna(False).astype(bool)   # no description: False

open_now = jobs[~jobs.is_repost & jobs.live]
print(f"4.5 Among the {len(open_now):,} original listings still open on September 30:")
display(pd.DataFrame({
    "tagged": [open_now.asks_5_plus_years.sum(), open_now.degree_no_equivalent.sum()],
    "share of open listings": [open_now.asks_5_plus_years.mean(), open_now.degree_no_equivalent.mean()],
    "listings where it is unknown": [open_now.years_required.isna().sum(), open_now.degree_required.isna().sum()],
}, index=["asks 5+ years", "degree required, no equivalent"]).style.format({"tagged": "{:,.0f}", "share of open listings": "{:.1%}", "listings where it is unknown": "{:,.0f}"}))

**What the data shows**

- **Manager level and above is 13.0% of original listings and 11.2% of those still open.** The levels ask a median 1 year (entry), 5 (mid_senior), 6 (manager) and 10 (leadership), so a career changer without management experience mostly can't get these listings. It is a tag because the person decides, and the app's job-level filter already lets them.
- **Open 90+ days: 31,816 of the 209,565 listings still open (15.2%)** are tagged and ranked lower. A further 7,229 (3.4%) are likely ghosts and hidden.
- **Staffing agencies and recruiters are 1.9% of listings** (679 companies: 126 by name or website, 553 by about text only). Their listings close in a median 13.1 days against 23.6 for employers. **Recruiters aren't only staffing agencies:** among employers, recruiters with 5+ listings still close in 7.6 days, against 24.1 where no recruiter is named. The rule is a first-pass keyword rule, so it tags and never hides.
- **Senior titles are 36.4% of listings and mostly out of reach for a newcomer.** Within `mid_senior`, senior-titled listings ask a median 6 years against 4 (81% ask 5 or more, against 43%), are open to career changers 4.8% of the time against 16.5%, and pay $77.81 an hour against $56.25. They stay visible, ranked lower, with a "senior" tag.
- **Text tags:** 32.5% of the open listings ask for 5+ years (another 103,399 state no years), and only 2.6% require a degree with no equivalent.

## 5. Keep: the role buckets

Everything the rules above don't tag or hide is kept. This section looks at what is kept: the role buckets (a bucket is a group of O*NET occupations that do similar work, such as "networks and systems"), how big each one is, and which are big enough to analyze on their own. The marketing-specialist tag comes first, because it changes the buckets.

### 5.1 Marketing specialists

O*NET 13-1161 ("market research analysts and marketing specialists") mixes two kinds of work. A listing whose title names analysis or research stays in **business and data analysis**, which the Data Analytics program trains for. The rest are **marketing specialists**: marketing work, not what that program trains for. The split is a title keyword rule nobody has hand-checked (decided 2026-10-13), so marketing specialists are a tag and an opt-in filter, and the listings stay.

In [ ]:
# 5.1 Split O*NET 13-1161 by title: analysts stay in business and data analysis, the rest become marketing specialists
from src.career_switch import MARKET_ANALYST_TITLE

print("Words that make a 13-1161 title an analyst role (anything without them is a marketing specialist):")
print(MARKET_ANALYST_TITLE)

jobs["role_bucket"] = jobs.role_bucket.astype(object)   # plain text, so a new bucket name can be written into it
is_marketing = marketing_specialist(jobs.onet_code, jobs.title).to_numpy()
jobs.loc[is_marketing, "role_bucket"] = MARKETING_BUCKET

originals = jobs[~jobs.is_repost]
market_research = originals[originals.onet_code.astype(str).str.startswith("13-1161")]
print(f"\n5.1 O*NET 13-1161: {len(market_research):,} original listings; "
      f"{(market_research.role_bucket == MARKETING_BUCKET).mean():.0%} are marketing specialists (tagged), "
      f"{(market_research.role_bucket != MARKETING_BUCKET).mean():.0%} stay in business and data analysis")

### 5.2 How big is each role bucket?

Original listings only (a repost would count the same job twice).

In [ ]:
# 5.2 Size and behavior of each role bucket (original listings)
originals = jobs[~jobs.is_repost]
buckets = originals.groupby("role_bucket").agg(
    listings=("id", "size"),
    still_open=("live", "sum"),
    companies=("company_id", "nunique"),
    entry_level=("level", lambda level: (level == "entry").mean()),
    median_hourly_pay=("salary_hourly_usd", "median"),
)
buckets["median days open"] = originals.groupby("role_bucket")[["days_open", "live"]].apply(median_days_open)
buckets = buckets.sort_values("listings", ascending=False)
print("5.2 Role buckets in the scope (original listings)")
display(buckets.style.format({"listings": "{:,.0f}", "still_open": "{:,.0f}", "companies": "{:,.0f}", "entry_level": "{:.1%}",
                              "median_hourly_pay": "${:,.2f}", "median days open": "{:.1f}"}))

### 5.3 Which buckets get their own analysis?

A bucket gets its own analysis, and its own paths in the retraining compass, when it passes **two tests**, both fixed before looking at results:

1. **Enough listings.** We want to say something about a bucket by source and by month. The margin of error on a share is at most ±5 points when a group has about 400 listings (1.96 × √(0.25 / 400) = 0.049), so we split a bucket as finely as we can while at least 80% of its groups have 400+ listings. The splits we try, finest first: month × source type, then source type, then month, then the whole bucket.
2. **Enough career changers.** At least 30 people in the résumé data (Profiles) moved into the bucket, counting all three groups: retrained after working in another field, retrained as a new graduate, and changed roles within tech.

A bucket that passes one test but not the other is analyzed together with a close bucket. Its listings always stay in the table.

In [ ]:
# 5.3 The two tests for each role bucket
MIN_LISTINGS_PER_GROUP, MIN_SHARE_OF_GROUPS, MIN_CHANGERS = 400, 0.80, 30

# Test 1: the finest split where at least 80% of the groups have 400+ listings (full months only)
SPLITS = {"month x source type": ["month", "source_type"], "source type": ["source_type"], "month": ["month"], "whole bucket": []}
full_months = originals[~originals.partial_month]
finest_split = {}
for bucket, listings in full_months.groupby("role_bucket"):
    finest_split[bucket] = "none"
    for name, columns in SPLITS.items():
        group_sizes = listings.groupby(columns, observed=True).size() if columns else pd.Series([len(listings)])
        if (group_sizes >= MIN_LISTINGS_PER_GROUP).mean() >= MIN_SHARE_OF_GROUPS:
            finest_split[bucket] = name
            break

# Test 2: career changers per bucket, from Profiles' frozen modeling table (counts only)
profiles = pd.read_pickle(DATA_DIR / "profiles" / "modeling_table.pkl")
changers = profiles[profiles.career_changer]
changers_by_pool = pd.crosstab(changers.target_bucket, changers.pool, margins=True, margins_name="all groups")
print("5.3 Career changers per role bucket, by group (Profiles)")
display(changers_by_pool.style.format("{:,.0f}"))

decision = pd.DataFrame({
    "listings": buckets.listings,
    "finest split": pd.Series(finest_split),
    "career changers": changers_by_pool["all groups"].drop("all groups").reindex(buckets.index).fillna(0),
})
decision["enough listings"] = decision["finest split"] != "none"
decision["enough career changers"] = decision["career changers"] >= MIN_CHANGERS
decision["verdict"] = np.where(decision["enough listings"] & decision["enough career changers"], "own analysis and compass paths",
                               "analyze with a close bucket")
print("\n5.3 Role buckets: own analysis, or analyzed with a close bucket (every listing stays either way)")
display(decision.style.format({"listings": "{:,.0f}", "career changers": "{:,.0f}"}))

**What the data shows**

- **O*NET 13-1161 is mostly marketing:** it has 23,292 original listings, and 64% (15,013) have no analyst, research or insights word in the title. They are tagged marketing specialists (28.1% entry level, $36.06 an hour) and stay in the table; the other 36% stay in business and data analysis. The title rule has not been hand-checked, which is why it tags and doesn't drop.
- **Four buckets are large:** software development (242,105 original listings), networks and systems (159,062), IT project management (106,253) and data science (98,782). Technical sales (8,965) is the smallest named bucket, and "other computer roles" (3,494) is a catch-all.
- **Eight buckets get their own analysis and compass paths.** Each has enough listings to split by source type or finer, and 38 (quality assurance, the thinnest) to 669 career changers. Business and data analysis, marketing specialists, technical sales and other computer roles have no career changers in the frozen profiles, so they are analyzed with a close bucket; their listings stay. IT project management is large but mostly reached from another tech role (189 of its 262 career changers).

## 6. The 2026 market by role bucket and program

The retraining compass joins two things: where people moved (from the résumé data, Profiles) and what is open now (from the listings). This section builds the listings side: for each role bucket and each TripleTen program, what is open, how fast listings close, and what they pay and ask. The next cell joins it to Profiles' paths.

One column needs a definition. **Visible to a newcomer** means a listing that isn't a likely ghost and isn't manager level or above, the listings a career changer without management experience would see in the app. The compass shows pay for that view, because it is the pay a newcomer can actually reach.

In [ ]:
# 6.1 The 2026 market for each role bucket (the eight computer and mathematical buckets plus technical sales and business and data analysis)
from src.career_switch import EXTRA_ORDER, PROGRAM_ORDER, ROLE_ORDER

jobs["cc_visible"] = ~jobs.likely_ghost & ~jobs.manager_or_above   # visible to a newcomer without management experience
FULL_WEEKS = jobs.loc[jobs.full_week, "first_seen"].dt.tz_localize(None).dt.to_period("W").nunique()   # weeks with a full seven days of data
originals = jobs[~jobs.is_repost]


def market_table(listings, by, order):
    """One row per group in `by`: what is open, how fast it closes, and what it pays and asks (original listings)."""
    table = listings.groupby(by).agg(
        open_now=("live", "sum"),
        new_per_week=("full_week", lambda full_week: full_week.sum() / FULL_WEEKS),
        median_hourly_pay=("salary_hourly_usd", "median"),
        entry_level=("level", lambda level: (level == "entry").mean()),
        open_to_career_changers=("career_open", "mean"),
        mentions_remote=("mentions_remote", "mean"),
    )
    table["median days open"] = listings.groupby(by)[["days_open", "live"]].apply(median_days_open)
    visible = listings[listings.cc_visible]
    table["median hourly pay, newcomer view"] = visible.groupby(by).salary_hourly_usd.median()
    return table.reindex(order)


MARKET_FORMATS = {"open_now": "{:,.0f}", "new_per_week": "{:,.0f}", "median_hourly_pay": "${:,.2f}", "entry_level": "{:.1%}",
                  "open_to_career_changers": "{:.1%}", "mentions_remote": "{:.1%}", "median days open": "{:.1f}",
                  "median hourly pay, newcomer view": "${:,.2f}"}

role_market = market_table(originals[originals.role_bucket.isin(ROLE_ORDER + EXTRA_ORDER)], "role_bucket", ROLE_ORDER + EXTRA_ORDER)
print("6.1 The 2026 market by role bucket (original listings)")
display(role_market.style.format(MARKET_FORMATS))

# Marketing specialists are left out of the program table: Data Analytics trains analysts, not marketing work
program_market = market_table(originals[originals.role_bucket != MARKETING_BUCKET], "tripleten_program", PROGRAM_ORDER)
print("\n6.1 The 2026 market by TripleTen program (adjacent tech role = no program trains for it)")
display(program_market.style.format(MARKET_FORMATS))

In [ ]:
# 6.2 The compass paths: where people moved (Profiles), joined to the 2026 market of the role they moved into
# A path is a "from" and a role (the bucket they moved into), with the number of people who made that move. Three kinds of "from":
# the career they left, the field they trained in, and (for moves within tech) the tech job they held before.
THIN = 10   # a path with fewer than 10 people is marked thin
retrained = changers[changers.pool.str.startswith("retrained")]
within_tech = changers[changers.pool.eq("changed roles within tech")]
PATH_KINDS = {"prior career": (retrained, "background_area"), "field trained in": (retrained, "retrain_field"),
              "tech job before the move": (within_tech, "from_bucket")}
paths = pd.concat([people.groupby([origin, "target_bucket"]).size().rename("people").reset_index()
                   .rename(columns={origin: "from"}).assign(path=kind)
                   for kind, (people, origin) in PATH_KINDS.items()], ignore_index=True)
paths["thin (under 10 people)"] = paths.people < THIN
compass_paths = paths.merge(role_market, left_on="target_bucket", right_index=True, how="left")

SHOWN = ["people", "thin (under 10 people)", "open_now", "median_hourly_pay", "median days open", "entry_level"]
for kind in PATH_KINDS:
    largest = compass_paths[compass_paths.path == kind].sort_values("people", ascending=False).head(5)
    print(f"6.2 The five largest paths by {kind}, with the 2026 market of the role they lead to")
    display(largest.set_index(["from", "target_bucket"])[SHOWN].style.format(
        {"people": "{:,.0f}", "open_now": "{:,.0f}", "median_hourly_pay": "${:,.2f}", "median days open": "{:.1f}", "entry_level": "{:.1%}"}))
print("\n6.2 Share of paths resting on fewer than 10 people")
display(compass_paths.groupby("path")["thin (under 10 people)"].mean().to_frame("share thin").style.format("{:.1%}"))

# Saved for ML_Fit, which assembles the compass. Written to data/clean/lean/ with this notebook's other tables.
role_market.to_pickle(LEAN_DIR / "role_market.pkl")
program_market.to_pickle(LEAN_DIR / "program_market.pkl")
compass_paths.to_pickle(LEAN_DIR / "compass_paths_2026.pkl")
print(f"\nsaved role_market.pkl ({len(role_market)} buckets), program_market.pkl ({len(program_market)} programs) "
      f"and compass_paths_2026.pkl ({len(compass_paths):,} paths)")

**What the data shows**

- **The 2026 market by bucket:** software development (66K open, 9.2K new a week, $73.54 an hour) and networks and systems (41K open) are the largest; IT support is the door (44% open to career changers, 22% entry level, $31.10); technical sales is the slowest to close (median 38 days) and the most likely to mention remote work (41% of descriptions).
- **By TripleTen program:** AI Software Engineering (63K open) and AI & Machine Learning pay the most (about $74); **Quality Assurance and Data Analytics are the most open to career changers** (27% and 26%), Data Analytics at $51.50 an hour now that marketing specialists are out of its market; adjacent tech roles hold 86K open listings.
- **Most compass paths are thin:** 78% of prior-career paths and 67% of training paths rest on fewer than 10 people (within-tech moves: 43%), so the compass always shows how many people a path rests on. The largest within-tech move is IT support to networks and systems (279 people).

## 7. The rules together, and the final table

### 7.1 How many listings does each rule touch?

For each rule: how many listings match, and how many of the original listings still open on September 30 (the ones a person would see in the app). Then the answer to the question behind the hide and tag rules: **how much of the scope is left?**

In [ ]:
# 7.1 What each rule touches, and how much is left
jobs["marketing_specialist"] = jobs.role_bucket.eq(MARKETING_BUCKET)
# The rules that have a column (the drop rules happen upstream), named like "Hide: likely ghost"
RULES_TABLE = {f"{r['treatment'].split()[0].strip(',')}: {r['rule'].lower()}": r["column"] for r in RULES if r["column"]}
rule_flags = pd.DataFrame({name: jobs[column] for name, column in RULES_TABLE.items()})
open_originals = ~jobs.is_repost & jobs.live

touched = pd.DataFrame({
    "all listings": rule_flags.sum(),
    "share of the scope": rule_flags.mean(),
    "open originals": rule_flags[open_originals].sum(),
    "share of open originals": rule_flags[open_originals].mean(),
})
print("7.1 Listings each rule touches (a listing can match several)")
display(touched.style.format({"all listings": "{:,.0f}", "share of the scope": "{:.1%}", "open originals": "{:,.0f}", "share of open originals": "{:.1%}"}))

# How much is left? Among the original listings still open on September 30:
open_now = jobs[open_originals]
visible = open_now[~open_now.likely_ghost]                                   # after the one hide rule
newcomer = open_now[~open_now.likely_ghost & ~open_now.manager_or_above]      # also without manager level and above (the app's default for an applicant with no management experience)
print(f"\n7.1 Of the {len(open_now):,} original listings still open on September 30:")
print(f"  {len(visible):,} ({len(visible) / len(open_now):.1%}) remain after the hide rule (likely ghosts)")
print(f"  {len(newcomer):,} ({len(newcomer) / len(open_now):.1%}) remain for an applicant with no management experience")
print(f"  an opt-in 'asks 5+ years' filter would remove another {newcomer.asks_5_plus_years.sum():,} of those "
      f"({newcomer.asks_5_plus_years.mean():.1%}); 'degree required, no equivalent' another {newcomer.degree_no_equivalent.sum():,} "
      f"({newcomer.degree_no_equivalent.mean():.1%})")

### 7.2 The final table

One row per in-scope listing, keyed by `id`: the role bucket, the level, and one column for each rule, so any later notebook (or the app) can join it to `EDA_SDA_1_Lean`'s in-scope table by `id` without copying that table's 70 columns. Nothing is removed: a listing a rule hides or tags is still a row.

In [ ]:
# 7.2 Save the final table: a row per listing, with the role bucket, the level and each rule's column
# What each column is. The rules' columns are described by the rules list in section 2 (one home for the definitions)
OTHER_COLUMNS = {
    "role_bucket": "The role bucket from the O*NET code; O*NET 13-1161 titles without an analysis word are marketing specialists (5.1)",
    "tripleten_program": "The TripleTen program that trains for the role, or 'adjacent tech role'",
    "level": "entry, mid_senior, manager or leadership, from PredictLeads' seniority label (4.1)",
    "cc_visible": "Visible to a newcomer without management experience: not a likely ghost and not manager level or above (6.1)",
}
RULE_COLUMNS = {r["column"]: f"{r['treatment'].split()[0].strip(',').upper()}: {r['rule']} (section {r['section']})" for r in RULES if r["column"]}
FLAG_COLUMNS = {**OTHER_COLUMNS, **RULE_COLUMNS}
final = jobs[["id", "company_id", *FLAG_COLUMNS]]
assert final.id.is_unique and len(final) == len(jobs)
final.to_parquet(LEAN_DIR / "scope_flags.parquet")

print("7.2 The final table's columns")
display(pd.DataFrame({"column": list(FLAG_COLUMNS), "what it is": list(FLAG_COLUMNS.values())}).style.hide(axis="index"))
print(f"saved scope_flags.parquet: {len(final):,} listings x {final.shape[1]} columns ({(LEAN_DIR / 'scope_flags.parquet').stat().st_size / 1e6:.0f} MB)")

**What the data shows**

- **One hide rule, and it touches little:** likely ghosts are 0.9% of the scope and 3.4% of the original listings still open on September 30. After it, 96.6% of the open listings remain.
- **An applicant with no management experience still sees 85.7% of the open listings** (179,562 of 209,565) once manager level and above, 11.2% of the open listings, is also set aside. The tags are far larger than the hide: 37.5% of the open listings have a senior title, and 15.2% are open 90+ days.
- **The text tags would hide the most if turned into filters:** an opt-in "asks 5+ years" filter would remove another 30.6% of what an applicant sees, while "degree required, no equivalent" would remove 2.5%. Staffing agencies and recruiters are 1.5% of the open listings, and marketing specialists 1.9%.
- **The final table is `scope_flags.parquet`:** 878,937 listings and 14 columns, keyed by `id`, so it joins to `EDA_SDA_1_Lean`'s in-scope table without copying it.

## 8. Conclusion and next steps

**Keep, tag, hide, drop.** `EDA_SDA_1_Lean` drops three occupations that aren't a tech path (document management specialists, GIS technologists and search marketing strategists, 29,354 listings), leaving the scope of 878,937 listings (821,589 originals) from 68,863 companies. This notebook then decides what the app does with each listing. **One rule hides:** likely ghosts. **Seven rules tag:** manager level and above, open 90+ days, staffing agency or recruiter, senior title, asks 5+ years, degree required with no equivalent, and marketing specialist (O*NET 13-1161 titles without an analysis word). Everything else is kept. Nothing in the scope is removed here.

### Answers to the clarifying questions

| Question | Answer |
|:--|:--|
| Which long-open listings are ghosts, and which are still hiring? | A company that has gone quiet separates them. **7,229 likely ghosts are hidden** (3.4% of the open listings); the other 31,816 listings open 90+ days (15.2%) stay visible, tagged and ranked lower. The age cutoff is a choice: about 30% of old open listings still close every month, so there is no natural break (3) |
| Which recruiter and agency listings do we tag, and are fast recruiters just staffing agencies? | 1.9% of listings (679 companies) are tagged as posted by a staffing agency or recruiter, and they close in a median 13.1 days against 23.6 for employers. Recruiters close fast even at employers (7.6 days for recruiters with 5+ listings against 24.1 where none is named), so the tag and the recruiter signal are not the same thing. The tag is a first-pass keyword rule, so it tags and never hides (4.3) |
| How much is left once the hiding rules are applied? | **96.6% of the open listings** after the hide rule, and **85.7% for an applicant with no management experience** (also setting aside manager level and above). An opt-in "asks 5+ years" filter would remove another 30.6%; "degree required, no equivalent" another 2.5% (7.1) |
| Which role buckets have enough listings and career changers? | All twelve have enough listings to split by source type or finer; eight also have 30+ career changers (38 to 669) and get their own analysis and compass paths. Business and data analysis, marketing specialists, technical sales and other computer roles have none in the frozen profiles, so they are analyzed with a close bucket (5.3) |
| What does the 2026 market look like, for the compass? | Software development (66K open, $73.54 an hour) and networks and systems (41K) are the largest; IT support is the door (44% open to career changers, $31.10); technical sales is the slowest to close (38 days). Most compass paths are thin, so the compass shows how many people each rests on (6) |

### For the career changer
- **IT support is the clearest door** (44% open, 22% entry level) at the lowest pay ($31 an hour); **Quality Assurance and Data Analytics** are the most open TripleTen programs (27% and 26%; Data Analytics at $51.50).
- **Senior-titled listings are mostly out of reach** (81% ask 5+ years, 4.8% open to newcomers): shown with a "senior" tag and ranked lower, not hidden.
- **Most paths are thin,** so the compass always shows how many people a path rests on. The largest within-tech move is IT support to networks and systems (279 people).

### What this notebook leaves to others
- **How listings behave** (company volume, source, freshness, pay disclosure, reposts, missing values) is answered in `EDA_SDA_1_Lean` (5, 6) and tested with the confounders held fixed in `EDA_SDA_4`.
- **The modeling columns** the original built (copy detection, the company-history, market and pay features, the company facts) and their screening move to `EDA_SDA_4_Lean`. The original `EDA_SDA_2` stays their source until then, so it can't be retired yet (`../ANALYSIS.md`).
- **The app's filter fields** (job type, work arrangement, schedule, title cleaning) are left out; the app builds what it needs.

### Next steps
- **EDA_SDA_3 (text):** the keyword rules here (staffing agency, marketing specialist) and in the text (years asked, degree) are unvalidated, so they stay tags and opt-in filters; clean descriptions from `scope_descriptions.parquet`; map skills to O*NET; add evergreen wording to the ghost check.
- **EDA_SDA_4 (statistics):** the behavior analysis and the feature screening, with the confounders held fixed.
- **Ask PredictLeads:** why job-board listings stop after May, and why descriptions go missing more often from June.